# Seminar 2: Tokenization & Model Mechanics

**Practice focus:** understand the main ideas behind how text becomes numerical input for an LLM and how self-attention creates context-aware token representations.

### Main workflow

**Text → tokens → numerical representation → embeddings + position → self-attention → contextual representation**

## 0. Importing the tools we need

We use only a few common Python packages in the main seminar:

- `numpy`: for numerical vectors and matrices
- `pandas`: for readable tables
- `scikit-learn`: for ready-made Bag-of-Words and TF–IDF tools

The optional RNN/LSTM appendix at the very end uses PyTorch, but it is **not required for the main seminar**.

In [1]:
# If needed, uncomment this line once:
# %pip install -q numpy pandas scikit-learn

import numpy as np
import pandas as pd

## 1. From text to tokens

A model cannot directly work with words. We first split text into smaller units called **tokens**.

For this first example we use a very simple method:

1. convert text to lowercase
2. remove one punctuation mark
3. split at spaces

Real LLM tokenizers are more sophisticated and often split words into **subwords**.

In [2]:
sentence = "Should we go to a pizzeria or do you prefer a restaurant?"

# Make the text easier to process
clean_sentence = sentence.lower().replace("?", "")

# Split the sentence into words
words = clean_sentence.split()

print("Original sentence:")
print(sentence)

print("\nTokens:")
print(words)

Original sentence:
Should we go to a pizzeria or do you prefer a restaurant?

Tokens:
['should', 'we', 'go', 'to', 'a', 'pizzeria', 'or', 'do', 'you', 'prefer', 'a', 'restaurant']


## 2. Vocabulary

The **vocabulary** is the list of unique tokens known in our small **corpus** (corpus = collection of documents).

In [3]:
vocabulary = []

for word in words:
    if word not in vocabulary:
        vocabulary.append(word)

# Sort only to make the output easier to read
vocabulary.sort()

print("Vocabulary:")
print(vocabulary)
print("\nVocabulary size:", len(vocabulary))

Vocabulary:
['a', 'do', 'go', 'or', 'pizzeria', 'prefer', 'restaurant', 'should', 'to', 'we', 'you']

Vocabulary size: 11


## 3. One-hot encoding

A **one-hot vector** has:

- one `1` at the position of the word,
- `0` everywhere else.

This represents **word identity**, but it does not represent meaning.

In [4]:
word = "restaurant"

# Start with a vector containing only zeros
one_hot = np.zeros(len(vocabulary), dtype=int)

# Find where "restaurant" is in the vocabulary
position = vocabulary.index(word)

# Put 1 at that position
one_hot[position] = 1

print("Word:", word)
print("One-hot vector:")
print(one_hot)

Word: restaurant
One-hot vector:
[0 0 0 0 0 0 1 0 0 0 0]


### One-hot encoding for the whole sentence

Now we repeat the same idea for every token.

Each **row** represents one token. Each **column** represents one vocabulary item.

In [5]:
one_hot_matrix = []

for word in words:
    vector = np.zeros(len(vocabulary), dtype=int)
    position = vocabulary.index(word)
    vector[position] = 1
    one_hot_matrix.append(vector)

one_hot_table = pd.DataFrame(
    one_hot_matrix,
    index=words,
    columns=vocabulary
)

one_hot_table

,a,do,go,or,pizzeria,prefer,restaurant,should,to,we,you
should,0,0,0,0,0,0,0,1,0,0,0
we,0,0,0,0,0,0,0,0,0,1,0
go,0,0,1,0,0,0,0,0,0,0,0
to,0,0,0,0,0,0,0,0,1,0,0
a,1,0,0,0,0,0,0,0,0,0,0
pizzeria,0,0,0,0,1,0,0,0,0,0,0
or,0,0,0,1,0,0,0,0,0,0,0
do,0,1,0,0,0,0,0,0,0,0,0
you,0,0,0,0,0,0,0,0,0,0,1
prefer,0,0,0,0,0,1,0,0,0,0,0


### Key limitation

Compare `restaurant` and `pizzeria`.

Humans know that they are related concepts, but one-hot encoding treats them as completely separate positions.

**One-hot encoding knows identity, not semantic similarity.**

## 4. Bag-of-Words (BoW)

One-hot encoding represents individual tokens.

**Bag-of-Words** represents a whole document by counting how many times each vocabulary word appears.

We use a small corpus similar to the lecture example.

- `Rows` = documents 
- `Columns` = vocabulary terms
- `Cells` = how many times a word appears in a document

In [6]:
corpus = [
    "this movie is awesome awesome",
    "i do not say it is good but it is awesome",
    "awesome only a fool can say that"
]

print(corpus)

['this movie is awesome awesome', 'i do not say it is good but it is awesome', 'awesome only a fool can say that']


### Building one shared vocabulary for all documents

In [7]:
all_words = []

for document in corpus:
    document_words = document.lower().split()

    for word in document_words:
        if word not in all_words:
            all_words.append(word)

all_words.sort()

print("Vocabulary:")
print(all_words)

Vocabulary:
['a', 'awesome', 'but', 'can', 'do', 'fool', 'good', 'i', 'is', 'it', 'movie', 'not', 'only', 'say', 'that', 'this']


### Counting words in each document

The idea is simple:

> For every document, count how often every vocabulary word appears.

In [8]:
# Creating empty list that will contain the numerical representation of every document
bow_matrix = []

# Processing the docs one by one, identifying individual words in each doc
for document in corpus:
    document_words = document.lower().split()
    # Empty list for the current doc
    counts = []

    # Goes thru every word in the complete vocabulary
    for vocabulary_word in all_words:
        # Most important part = counts how many times that specific vocab word occurs in the current document
        count = document_words.count(vocabulary_word)
        # Adds that number to the document's vector
        counts.append(count)

    # Adds the completed vector for the current document to the final BoW matrix
    bow_matrix.append(counts)

bow_table = pd.DataFrame(
    bow_matrix,
    index=["Document 1", "Document 2", "Document 3"],
    columns=all_words
)

bow_table

,a,awesome,but,can,do,fool,good,i,is,it,movie,not,only,say,that,this
Document 1,0,2,0,0,0,0,0,0,1,0,1,0,0,0,0,1
Document 2,0,1,1,0,1,0,1,1,2,2,0,1,0,1,0,0
Document 3,1,1,0,1,0,1,0,0,0,0,0,0,1,1,1,0


### Why is it called a *bag* of words?

Because word order is ignored.

These two sentences contain the same words:

- `dog bites man`
- `man bites dog`

Their meaning is very different, but their BoW vectors are identical.

In [9]:
sentence_1 = "dog bites man"
sentence_2 = "man bites dog"

small_vocabulary = ["bites", "dog", "man"]

for sentence in [sentence_1, sentence_2]:
    sentence_words = sentence.split()
    counts = []

    for word in small_vocabulary:
        counts.append(sentence_words.count(word))

    print(sentence, "->", counts)

dog bites man -> [1, 1, 1]
man bites dog -> [1, 1, 1]


## 5. TF–IDF

BoW counts every word equally. But some words are more informative than others.

**TF–IDF** gives a higher weight to a term when it is:

- frequent in one document, **and**
- relatively rare in the other documents.

We let `scikit-learn` perform the calculation and focus on interpreting the result.

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Creating the TF-IDF calculator
vectorizer = TfidfVectorizer()

# Learning the vocabulary and calculate TF-IDF values
TFIDF = vectorizer.fit_transform(corpus)

# Converting the result to a normal table
TFIDF_table = pd.DataFrame(
    TFIDF.toarray(),
    index=["Document 1", "Document 2", "Document 3"],
    columns=vectorizer.get_feature_names_out()
)

TFIDF_table.round(2)

,awesome,but,can,do,fool,good,is,it,movie,not,only,say,that,this
Document 1,0.59,0.0,0.00,0.0,0.00,0.0,0.38,0.0,0.5,0.0,0.00,0.00,0.00,0.5
Document 2,0.18,0.3,0.00,0.3,0.00,0.3,0.45,0.6,0.0,0.3,0.00,0.23,0.00,0.0
Document 3,0.27,0.0,0.45,0.0,0.45,0.0,0.00,0.0,0.0,0.0,0.45,0.34,0.45,0.0


### Compare BoW and TF–IDF

- **BoW:** How many times does the word appear?
- **TF–IDF:** How informative is the word in this document relative to the corpus?

Both methods still have an important limitation:

> They do not truly understand meaning or context.

## Mini-exercise 1: BoW vs. TF–IDF

Work in pairs. Before running the next cell, **predict the result**.

We have three short documents:

- `cats are friendly pets`
- `dogs are friendly pets`
- `cats chase mice`

Questions:

1. Which words do you expect to have the highest **BoW counts**?
2. Which words should receive relatively high **TF–IDF values** because they are more distinctive?
3. Run the code and check your prediction.
4. Then edit the third sentence so that it also contains the word `friendly`. Run the cell again. What happens to the TF–IDF importance of `friendly`?

In [11]:
from sklearn.feature_extraction.text import CountVectorizer

exercise_corpus = [
    "cats are friendly pets",
    "dogs are friendly pets",
    "cats chase mice"
]

# --- Bag-of-Words ---
bow_exercise = CountVectorizer()
bow_result = bow_exercise.fit_transform(exercise_corpus)

print("BoW table:")
display(pd.DataFrame(
    bow_result.toarray(),
    columns=bow_exercise.get_feature_names_out()
))

# --- TF-IDF ---
idf_exercise = TfidfVectorizer()
idf_result = idf_exercise.fit_transform(exercise_corpus)

print("TF-IDF table:")
display(pd.DataFrame(
    idf_result.toarray(),
    columns=idf_exercise.get_feature_names_out()
).round(2))


BoW table:


,are,cats,chase,dogs,friendly,mice,pets
0,1,1,0,0,1,0,1
1,1,0,0,1,1,0,1
2,0,1,1,0,0,1,0


TF-IDF table:


,are,cats,chase,dogs,friendly,mice,pets
0,0.50,0.50,0.00,0.0,0.50,0.00,0.50
1,0.46,0.00,0.00,0.6,0.46,0.00,0.46
2,0.00,0.47,0.62,0.0,0.00,0.62,0.00


## 6. Modern tokenization: subwords

Modern LLMs usually do not store every possible full word.

Instead, words can be divided into reusable pieces.

For example:

`unhappiness → un + happi + ness`

This is especially useful for rare words, typos, names and morphologically rich languages such as Hungarian or Turkish.

The important trade-off:

- `smaller vocabulary` → more subword pieces → longer sequences
- `larger vocabulary` → fewer pieces → bigger embedding matrix

In [12]:
# These examples are written manually only to illustrate the idea.

examples = {
    "unhappiness": ["un", "happi", "ness"],
    "playing": ["play", "ing"],
    "transformers": ["transform", "ers"],
    "érthetetlen": ["ért", "het", "etlen"]
}

for word in examples:
    print(word, "->", examples[word])

unhappiness -> ['un', 'happi', 'ness']
playing -> ['play', 'ing']
transformers -> ['transform', 'ers']
érthetetlen -> ['ért', 'het', 'etlen']


## 7. Embeddings

One-hot vectors are large and sparse.

An **embedding** represents a token with a smaller dense vector, for example:

`cat → [0.12, -0.45, 0.33, ...]`

In a real LLM, these vectors are **learned during training**.

Here we use small made-up vectors so we can clearly see the idea.

In [13]:
embeddings = {
    "cat": np.array([0.9, 0.8, 0.1]),
    "dog": np.array([0.8, 0.9, 0.1]),
    "bank": np.array([0.2, 0.1, 0.9])
}

print("cat:", embeddings["cat"])
print("dog:", embeddings["dog"])
print("bank:", embeddings["bank"])

cat: [0.9 0.8 0.1]
dog: [0.8 0.9 0.1]
bank: [0.2 0.1 0.9]


### Semantic similarity intuition

`cat` and `dog` have similar vectors in this toy example.

We can measure similarity with a **dot product**. For today, you only need the intuition:

> Similar vectors tend to produce larger similarity scores.

In [14]:
cat_dog = np.dot(embeddings["cat"], embeddings["dog"])
cat_bank = np.dot(embeddings["cat"], embeddings["bank"])

print("Similarity: cat vs dog =", round(cat_dog, 2))
print("Similarity: cat vs bank =", round(cat_bank, 2))

Similarity: cat vs dog = 1.45
Similarity: cat vs bank = 0.35


## Mini-exercise 2: Exploring embedding similarity

The vectors below are small **toy embeddings**. They are not learned from a real model, but they let us see the idea clearly.

Before running the code:

1. Which pair should be more similar: `apple–banana` or `apple–car`?
2. Run the cell and compare the dot products.
3. Change the vector for `car` from `[0.1, 0.2, 0.9]` to `[0.8, 0.7, 0.2]` and run it again.
4. What happened to the similarity score? Why?

In [15]:
exercise_embeddings = {
    "apple": np.array([0.9, 0.8, 0.1]),
    "banana": np.array([0.8, 0.7, 0.2]),
    "car": np.array([0.1, 0.2, 0.9])
}

apple_banana = np.dot(
    exercise_embeddings["apple"],
    exercise_embeddings["banana"]
)

apple_car = np.dot(
    exercise_embeddings["apple"],
    exercise_embeddings["car"]
)

print("apple vs banana:", round(apple_banana, 2))
print("apple vs car:", round(apple_car, 2))


apple vs banana: 1.3
apple vs car: 0.34


## 8. Positional information

Embeddings tell the model **what** the tokens are.

The transformer also needs to know **where** each token occurs.

A simple conceptual representation is:

- first token → position 0,
- second token → position 1,
- third token → position 2.

Real transformers use more sophisticated positional representations, but this simple example shows the purpose.

In [16]:
sentence_tokens = ["the", "cat", "sleeps"]

for position in range(len(sentence_tokens)):
    token = sentence_tokens[position]
    print("Token:", token, "| Position:", position)

Token: the | Position: 0
Token: cat | Position: 1
Token: sleeps | Position: 2


### Original transformer idea: add position to the embedding

Conceptually:

**input representation = token embedding + positional encoding**

The exact sine/cosine mathematics from the lecture is important theoretically, but we do not implement the full formula in this beginner seminar.

## 9. Self-attention: the central transformer idea

Self-attention answers:

> **Which other tokens are relevant to the current token, and how much information should be taken from them?**

In a real transformer, token embeddings may contain hundreds or thousands of dimensions, and the model learns separate **Query (Q), Key (K), and Value (V)** representations.

For this beginner demonstration we deliberately simplify the problem:

- only **3 tokens**
- only **2 dimensions per token**
- we use the same vectors for Q, K, and V, so conceptually **Q = K = V = X**

This lets us see the logic of attention without hiding it behind large matrices.

### The three steps

**1. Compare tokens → 2. Convert similarities into attention weights → 3. Combine information**

In [17]:
# Three small token vectors
#
# Each ROW represents one token.
# Each COLUMN represents one numerical feature/dimension.
#
# These numbers are chosen manually for teaching.
# In a real LLM, the representations are learned during training.

X = np.array([
    [1.0, 0.0],   # token 1
    [0.8, 0.2],   # token 2 - deliberately similar to token 1
    [0.0, 1.0]    # token 3 - deliberately different
])

print("Token matrix X:")
print(X)
print("\nShape of X:", X.shape)   # 3 tokens × 2 dimensions

Token matrix X:
[[1.  0. ]
 [0.8 0.2]
 [0.  1. ]]

Shape of X: (3, 2)


### Step 1: comparing every token with every other token

We want a score for:

- token 1 vs token 1, token 2, token 3
- token 2 vs token 1, token 2, token 3
- token 3 vs token 1, token 2, token 3

`X` has shape **3 × 2**: 3 token vectors, each with 2 dimensions.

`X.T` means **transpose**: rows become columns. Therefore `X.T` has shape **2 × 3**.

This makes the multiplication possible:

**(3 × 2) @ (2 × 3) → (3 × 3)**

The result is a 3 × 3 table containing one similarity score for every token pair.

Each score is a **dot product**: multiply matching vector elements and add them.

For example:

`token 1 · token 2 = (1.0 × 0.8) + (0.0 × 0.2) = 0.8`

In [18]:
# Comparing every token vector with every other token vector
scores = X @ X.T

print("X shape:   ", X.shape)
print("X.T shape: ", X.T.shape)
print("Scores:    ", scores.shape)

scores_table = pd.DataFrame(
    scores,
    index=["token 1", "token 2", "token 3"],
    columns=["token 1", "token 2", "token 3"]
)

scores_table.round(2)

X shape:    (3, 2)
X.T shape:  (2, 3)
Scores:     (3, 3)


,token 1,token 2,token 3
token 1,1.0,0.80,0.0
token 2,0.8,0.68,0.2
token 3,0.0,0.20,1.0


### Step 2: converting similarity scores into attention weights

The raw scores are useful for comparison, but they are not yet easy to interpret as proportions.

We therefore apply **softmax separately to each row**.

Softmax:

1. turns every score into a positive number,
2. gives relatively more weight to larger scores,
3. normalizes the row so that its values add up to **1**.

For example, if token 1 receives weights:

`[0.46, 0.37, 0.17]`

we can interpret this as token 1 taking approximately:

- 46% of its information from token 1,
- 37% from token 2,
- 17% from token 3.

The percentages are an intuition for the weights; mathematically they are normalized coefficients.

In [19]:
# A small softmax function

def softmax(row):
    # Exponentiation makes all values positive
    # and emphasizes larger scores
    numbers = np.exp(row)

    # Divide by the total so the row sums to 1
    return numbers / numbers.sum()


attention_weights = []

# Apply softmax to each token's row of similarity scores
for row in scores:
    attention_weights.append(softmax(row))

attention_weights = np.array(attention_weights)

attention_table = pd.DataFrame(
    attention_weights,
    index=["token 1", "token 2", "token 3"],
    columns=["token 1", "token 2", "token 3"]
)

attention_table.round(2)

,token 1,token 2,token 3
token 1,0.46,0.37,0.17
token 2,0.41,0.36,0.23
token 3,0.20,0.25,0.55


### Step 3: combining information from the tokens

Now we know **how much attention** each token gives to every token.

The final attention step uses these weights to create a **new representation** for each token.

For token 1, imagine its attention weights are approximately:

`[0.46, 0.37, 0.17]`

and the original vectors are:

- token 1 = `[1.0, 0.0]`
- token 2 = `[0.8, 0.2]`
- token 3 = `[0.0, 1.0]`

The new token-1 vector is a weighted mixture:

`0.46 × token 1 + 0.37 × token 2 + 0.17 × token 3`

So the new representation of token 1 no longer contains only its own information: it also contains information from the other tokens according to their relevance.

This is the essential purpose of self-attention:

> **turn an isolated token representation into a context-aware token representation.**

In [20]:
# Combining the token vectors using the calculated attention weights
attention_output = attention_weights @ X

attention_output_table = pd.DataFrame(
    attention_output,
    index=["token 1", "token 2", "token 3"],
    columns=["dimension 1", "dimension 2"]
)

attention_output_table.round(2)

,dimension 1,dimension 2
token 1,0.76,0.24
token 2,0.70,0.30
token 3,0.40,0.60


### What should you remember about self-attention?

The matrix mathematics supports one simple idea:

**1. Compare tokens → 2. calculate relevance weights → 3. combine information**

As a result, each token becomes **context-aware**.

For example, the representation of the word `bank` can become different in:

- `the bank approved the loan`
- `we sat on the bank of the river`

because the surrounding tokens are different.

### Connection to a real transformer

Our teaching example effectively assumes:

`Q = K = V = X`

A real transformer instead learns separate transformations:

`Q = XW_Q`  
`K = XW_K`  
`V = XW_V`

and calculates:

`Attention(Q, K, V) = softmax(QKᵀ / √dₖ)V`

You do **not** need to implement these learned matrices today. The goal is to understand the three-step attention logic.

## Mini-exercise 3: Changing attention by changing a token

Now test the central attention idea: **similar tokens receive higher relevance scores**.

The three vectors below represent three tokens.

1. Run the cell and inspect the attention weights.
2. For `token 1`, which other token receives more attention: `token 2` or `token 3`?
3. Change `token 3` from `[0.0, 1.0]` to `[0.9, 0.1]`.
4. Run the cell again. How do the attention weights for `token 1` change?

Focus on the pattern, not the matrix mathematics.

In [21]:
exercise_X = np.array([
    [1.0, 0.0],   # token 1
    [0.8, 0.2],   # token 2
    [0.0, 1.0]    # token 3 -- try changing this to [0.9, 0.1]
])

exercise_scores = exercise_X @ exercise_X.T

exercise_weights = []
for row in exercise_scores:
    exercise_weights.append(softmax(row))

exercise_weights = np.array(exercise_weights)

pd.DataFrame(
    exercise_weights,
    index=["token 1", "token 2", "token 3"],
    columns=["token 1", "token 2", "token 3"]
).round(2)

,token 1,token 2,token 3
token 1,0.46,0.37,0.17
token 2,0.41,0.36,0.23
token 3,0.20,0.25,0.55


## 10. Why did transformers replace RNNs and LSTMs?

This section is mainly **historical context**, not a new programming task.

Before transformers, language sequences were commonly processed with **RNNs** and **LSTMs**:

**RNN / LSTM**

`token 1 → token 2 → token 3 → token 4`

Information is passed forward **step by step**.

LSTMs improved the memory of basic RNNs, but processing remained sequential. This made long-distance relationships harder to learn and limited parallel computation.

**Transformer**

`token 1 ↔ token 2 ↔ token 3 ↔ token 4`

Self-attention allows tokens to evaluate the relevance of other tokens much more directly.

### Main takeaway

> **RNNs/LSTMs pass information sequentially; transformers use attention to connect tokens directly.**

This is the conceptual reason RNNs and LSTMs are mentioned in Lecture 2.

## 11. Summary

Try to explain these without looking back:

1. Why must text be converted into numbers?
2. What is the difference between **one-hot encoding** and **Bag-of-Words**?
3. What does **TF–IDF** add to raw word counts?
4. Why are **embeddings** useful?
5. Why does a transformer need **positional information**?
6. What are the three conceptual steps of **self-attention**?
7. Why does self-attention create **context-aware** token representations?
8. What is the main conceptual difference between transformers and RNN/LSTM sequence processing?

### Main seminar workflow

**Text → tokens → numerical representation → embeddings + position → self-attention → contextual representation**

### Most important self-attention takeaway

**Compare → weight → combine**

The purpose of the mathematics is to let every token decide which information from the current sequence is most relevant to its new representation.